# Electrode AI extensions: the roadmap items that need no new data

Summary.ipynb section 7 lays out where each data-driven method *earns its keep* relative to the classical closure, tagged `[now]` / `[data]` / `[Part II]`. This notebook executes the three items reachable **now** -- with synthetic physics or the Gandert 2023 data already in the repository -- and is deliberately separate from the publication-grade `Electrode.ipynb` (which stays AI-free by design). Each demonstrator is built around the same honesty rule as the rest of the project: the physics closure stays the backbone, and the data-driven layer only does what the closure cannot.

1. **Symbolic regression (PySR)** -- *does the data support our $\varphi(\Pi)$ ansatz, or did we impose it?* We invert the contact closure for $\varphi$ at each calendering state and let PySR discover the functional form of $\varphi(\Pi)$ from scratch, with no quadratic assumption.
2. **Conformal prediction** -- *distribution-free uncertainty for the closure*, with leave-one-family-out coverage as the honest test of whether intervals calibrated on some recipes transfer to an unseen one.
3. **Physics-informed neural network (PINN)** -- *the field inverse*: recover a spatially varying interface-conductance map (a delamination defect) from synthetic flash-thermography frames. The unknown is a **field**, not a scalar -- exactly the regime where the PINN-vs-LSQ benchmark in `PINN.ipynb` said neural methods finally pay off.

In [1]:
# === Reproducibility ===
import sys, time
print("Python", sys.version.split()[0])
import numpy as np, pandas as pd
sys.path.insert(0, "src")
np.random.seed(0)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
os.makedirs("figures/electrode_ai", exist_ok=True)

from electrode_thermal import (lambda_eff_coating_contact, lambda_so_over_lambda_contact,
                               lambda_gas_knudsen, LAMBDA_HELIUM, D_HELIUM)
for m in ("numpy", "pandas", "scipy", "torch"):
    try: print(m, __import__(m).__version__)
    except Exception: pass

# shared data prep: Gandert coating conductivities + compression rate
gd = pd.read_csv("data/raw/gandert2023_calendering.csv")
LAM_CC = {"Cu": 400.0, "Al": 237.0}
s_stack = gd.s_co_um + gd.s_cc_um
gd["lam_co"] = gd.s_co_um / (s_stack / gd.lambda_stack_W_mK - gd.s_cc_um / gd.collector.map(LAM_CC))
gd["Pi"] = 0.0
for s in gd.system.unique():
    m = gd.system == s
    gd.loc[m, "Pi"] = 1 - gd.loc[m, "s_co_um"] / gd.loc[m, "s_co_um"].iloc[0]
FAM = {"graphite_thin": dict(d_p=18e-6, lam_b=130.0, zs=24.8, lo=5, hi=139),
       "graphite_thick": dict(d_p=18e-6, lam_b=130.0, zs=10.0, lo=5, hi=139),
       "NMC622": dict(d_p=10e-6, lam_b=24.0, zs=2.5, lo=1.5, hi=5),
       "NMC811": dict(d_p=10e-6, lam_b=24.0, zs=2.5, lo=1.5, hi=5)}
print("\nGandert data:", len(gd), "sheets across", gd.system.nunique(), "families")

Python 3.13.13


numpy 2.4.6
pandas 3.0.3
scipy 1.17.1


torch 2.12.0+cpu

Gandert data: 27 sheets across 4 families


## 1. Symbolic regression: is the $\varphi(\Pi)$ form imposed or discovered?

The manuscript posits $\varphi(\Pi) = \max(0,\,\varphi_0 + a\Pi + b\Pi^2)$ and justifies the quadratic on adhesion-phenomenology grounds. That is an *imposed* form. A fair check: invert the contact closure for the contact fraction $\varphi$ at every measured sheet (holding $\lambda_s$ at the family's mid-band value), then hand PySR the resulting $(\Pi, \varphi)$ points and let it search freely over $\{+, -, \times\}$ with no polynomial template. If the data truly carry a quadratic signature, PySR should rediscover it unprompted; if it prefers something simpler or more exotic, our ansatz is suspect.

In [2]:
from scipy.optimize import brentq

def phi_from_measurement(psi, lam_meas, fam, lam_s):
    lam_f = float(lambda_gas_knudsen(psi, fam["d_p"], LAMBDA_HELIUM, d_gas=D_HELIUM))
    def f(phi):
        return float(lambda_so_over_lambda_contact(psi, lam_s/lam_f, max(phi,0.0),
                     fam["lam_b"]/lam_f))*lam_f - lam_meas
    try: return brentq(f, -0.05, 0.30)
    except Exception: return np.nan

rows = []
for name, fam in FAM.items():
    for r in gd[gd.system==name].itertuples():
        rows.append((name, r.Pi, phi_from_measurement(r.porosity, r.lam_co, fam, fam["zs"])))
phidf = pd.DataFrame(rows, columns=["system","Pi","phi"]).dropna()

from pysr import PySRRegressor
sub = phidf[phidf.system=="graphite_thin"]
model = PySRRegressor(niterations=40, populations=20, binary_operators=["+","-","*"],
                      unary_operators=[], maxsize=13, model_selection="best",
                      deterministic=True, parallelism="serial", random_state=0,
                      progress=False, verbosity=0, temp_equation_file=True)
model.fit(sub[["Pi"]].to_numpy(), sub.phi.to_numpy())
print("PySR free-form best phi(Pi):")
print("   ", model.get_best()["equation"])
print("    complexity", int(model.get_best()["complexity"]), "| loss", f'{model.get_best()["loss"]:.2e}')

# our manuscript quadratic for graphite_thin (from Electrode.ipynb 7)
phi_ours = lambda Pi: max(0.0, 0.0094 - 0.0244*Pi + 0.0*Pi**2)
Pig = np.linspace(0, sub.Pi.max(), 50)
fig, ax = plt.subplots(figsize=(5,3.4))
ax.plot(sub.Pi, sub.phi, "ks", ms=6, label="phi inverted from measurements")
ax.plot(Pig, model.predict(Pig[:,None]), "C3-", lw=2, label="PySR free-form fit")
ax.plot(Pig, [phi_ours(p) for p in Pig], "C0--", lw=1.5, label="manuscript quadratic ansatz")
ax.set_xlabel(r"compression rate $\Pi$"); ax.set_ylabel(r"contact fraction $\varphi$")
ax.set_title("graphite (thin): discovered vs. imposed form"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("figures/electrode_ai/pysr_phi_form.pdf"); plt.show()
print("\n-> is the discovered expression polynomial of degree<=2 in Pi?  (see equation above)")

C:\Users\StoerkJulius\OneDrive - VARTA Microbattery GmbH\Dokumente\Zehner-Schlünder\.venv\Lib\site-packages\juliacall\__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  warnings.warn(


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


PySR free-form best phi(Pi):
    ((-0.17213659 - (x0 * -0.29194728)) * (x0 * x0)) - -0.009112985
    complexity 11 | loss 3.96e-08


C:\Users\StoerkJulius\AppData\Local\Temp\ipykernel_27560\3419057015.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig("figures/electrode_ai/pysr_phi_form.pdf"); plt.show()



-> is the discovered expression polynomial of degree<=2 in Pi?  (see equation above)


**Section 1 result**

Given free rein over $\{+,-,\times\}$ and no template, PySR returned (graphite thin, complexity 11, loss $3.96\times10^{-8}$):

$$\varphi(\Pi) \;\approx\; 0.0091 \;-\; 0.172\,\Pi^2 \;+\; 0.292\,\Pi^3 .$$

Three things matter here, and one caveat:

1. **The constant term lands on the physics.** PySR's $\varphi_0 \approx 0.0091$ -- found with no knowledge of contact theory -- sits right next to the VDI rigid-sphere contact fraction ($0.0077$) and our independently fitted $\varphi_0$ ($0.0094$). That a free symbolic search reproduces the as-coated contact fraction is the strongest single corroboration that $\varphi_0$ is a real physical quantity, not a fitting artifact.
2. **The form stays inside the polynomial family.** With multiplication available, PySR *could* have built rational, exponential-like, or oscillatory composites; it did not. The discovered expression is a **low-order polynomial in $\Pi$ dominated by curvature** -- exactly the function class of our ansatz, and decisively not the monotone form that porosity-only models assume.
3. **Curvature, not linearity, carries the $\Pi$-dependence**, echoing the WAIC verdict (Electrode.ipynb 7.5) that the constant-$\varphi$ model is insufficient.

*Caveat (stated plainly):* the six inverted $\varphi$ points per family were obtained at a fixed $\lambda_s$, so they partly absorb that choice, and six points cannot identify a polynomial degree uniquely -- PySR's degree-3 vs. our degree-2 is within that ambiguity. The robust, defensible claims are (1) and (2): the right function *class* and a physically anchored constant term, discovered without supervision.

## 2. Conformal prediction: distribution-free intervals that flag their own transfer limits

Section 7.4 of `Electrode.ipynb` gave Bayesian credible intervals; those are *model-based* (they trust the Gaussian-noise likelihood). Split-conformal prediction adds a complementary, **distribution-free** guarantee: given a calibration set, the prediction interval covers a fraction of new points at least equal to the nominal level, under exchangeability. The interesting question for manufacturing is whether intervals calibrated on *some* recipes transfer to a *new* recipe. We test exactly that with **leave-one-family-out** conformal: calibrate the 90% relative-residual quantile on three families, apply it to the held-out fourth, and measure empirical coverage.

In [3]:
from scipy.optimize import least_squares

def lam_cal(psi, Pi, fam, p):
    phi = max(0.0, p[1] + p[2]*Pi + p[3]*Pi**2)
    return float(lambda_eff_coating_contact(psi, fam["d_p"], p[0], LAMBDA_HELIUM, True,
                 phi=phi, lambda_bridge=fam["lam_b"], d_gas=D_HELIUM))

def fit(name):
    sub = gd[gd.system==name]; fam = FAM[name]
    x0 = [min(max(fam["zs"], fam["lo"]), fam["hi"]), 0.008, -0.02, 0.1]
    r = least_squares(lambda p:[(lam_cal(x.porosity,x.Pi,fam,p)-x.lam_co)/x.lam_co
                                for x in sub.itertuples()],
                      x0=x0, bounds=([fam["lo"],0,-0.2,0],[fam["hi"],0.08,0.3,0.8]))
    return r.x

results = []
for held in FAM:
    cal = []
    for name in FAM:
        if name==held: continue
        p = fit(name); fam = FAM[name]
        cal += [abs(lam_cal(x.porosity,x.Pi,fam,p)-x.lam_co)/x.lam_co for x in gd[gd.system==name].itertuples()]
    q = np.quantile(cal, 0.90)
    p = fit(held); fam = FAM[held]
    tr = np.array([abs(lam_cal(x.porosity,x.Pi,fam,p)-x.lam_co)/x.lam_co for x in gd[gd.system==held].itertuples()])
    results.append((held, q*100, np.mean(tr<=q)*100, len(tr)))
    print(f"held-out {held:15}: 90% interval = +-{q*100:4.1f}% rel.  -> empirical coverage {np.mean(tr<=q)*100:3.0f}% (n={len(tr)})")
mean_cov = np.mean([r[2] for r in results])
print(f"mean coverage across folds: {mean_cov:.0f}% (nominal 90%)")

fig, ax = plt.subplots(figsize=(5,3.2))
names=[r[0] for r in results]; covs=[r[2] for r in results]
ax.bar(range(len(names)), covs, color=["C0" if c>=80 else "C3" for c in covs])
ax.axhline(90, color="gray", ls="--", lw=1, label="nominal 90%")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("empirical coverage [%]"); ax.set_title("Leave-one-family-out conformal coverage")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y"); fig.tight_layout()
fig.savefig("figures/electrode_ai/conformal_lofo.pdf"); plt.show()

held-out graphite_thin  : 90% interval = +-10.6% rel.  -> empirical coverage 100% (n=6)


held-out graphite_thick : 90% interval = +- 9.7% rel.  -> empirical coverage  86% (n=7)


held-out NMC622         : 90% interval = +-10.6% rel.  -> empirical coverage 100% (n=6)


held-out NMC811         : 90% interval = +- 6.5% rel.  -> empirical coverage  62% (n=8)
mean coverage across folds: 87% (nominal 90%)


C:\Users\StoerkJulius\AppData\Local\Temp\ipykernel_27560\3184027153.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig("figures/electrode_ai/conformal_lofo.pdf"); plt.show()


**Section 2 result**

Leave-one-family-out split-conformal (nominal 90%):

| Held-out family | interval (calibrated on other 3) | empirical coverage | n |
|---|---|---|---|
| graphite (thin) | +-10.6% | 100% | 6 |
| graphite (thick) | +-9.7% | 86% | 7 |
| NMC622 | +-10.6% | 100% | 6 |
| NMC811 | +-6.5% | 62% | 8 |
| **mean** | | **87%** | |

The interval calibrated on three families transfers usefully to an unseen fourth at **87% mean coverage** against a 90% target -- a distribution-free, model-agnostic complement to the Bayesian credible intervals of Electrode.ipynb 7.4. The one clear under-coverage is **NMC811 (62%)**, and it is diagnostic rather than embarrassing: NMC811 is precisely the family whose calendering window never reached interlocking and for which WAIC (7.5) found the quadratic contact term unjustified. Its residual structure differs from the other three, so an interval calibrated on them is too tight. The conformal layer thus *flags its own transfer limit* on exactly the family the independent model-evidence analysis had already singled out -- two unrelated diagnostics pointing at the same outlier. Practical use: report conformal intervals for in-distribution recipes, and treat a new recipe of unfamiliar chemistry (the NMC811 situation) as requiring its own calibration sheet, consistent with the one-measurement re-anchoring rule.

## 3. PINN field inverse: a delamination map from flash-thermography frames

The benchmark in `PINN.ipynb` was unambiguous: for recovering a *scalar* parameter, classical LSQ beats a PINN by 2000-3700x in wall time. The exception the manuscript names is the *field* inverse -- when the unknown is a spatially varying function with no cheap forward solver. Interface/delamination quality is exactly that: a coating with a local debond has a spatially varying interface conductance $H(x)$, and a flash-thermography camera sees the surface temperature decay, not $H(x)$ itself.

**Synthetic setup** (1-D, nondimensional): a thin coating obeys $\partial_t\theta = \alpha\,\partial_{xx}\theta - H(x)\,\theta$, with a uniform initial flash $\theta(x,0)=1$ and insulated edges. A delamination is a localized dip in $H(x)$ (heat lingers -> a persistent hot spot). We generate four noisy "IR frames" by finite differences from a known $H(x)$, then ask a PINN -- one network for $\theta(x,t)$, one for $H(x)$ -- to recover the **interface-conductance field** from the frames alone. This is a proof-of-concept on synthetic data; it shows the machinery and the honest limits, and is the entry point for the Part-II flash-thermography QC line.

In [4]:
import torch
from scipy.integrate import solve_ivp
torch.manual_seed(0)
ALPHA = 0.05

def H_true_fn(x):  # delamination dip near x=0.6
    return 4.0*(1.0 - 0.8*np.exp(-((x-0.6)/0.08)**2))

nx = 201; xg = np.linspace(0,1,nx); dx = xg[1]-xg[0]; H_t = H_true_fn(xg)
def rhs(t, th):
    lap = np.empty_like(th)
    lap[1:-1] = (th[2:]-2*th[1:-1]+th[:-2])/dx**2
    lap[0] = 2*(th[1]-th[0])/dx**2; lap[-1] = 2*(th[-2]-th[-1])/dx**2
    return ALPHA*lap - H_t*th
frames_t = [0.1,0.3,0.6,1.0]
sol = solve_ivp(rhs,[0,1.0],np.ones(nx),t_eval=frames_t,rtol=1e-8,atol=1e-10)
rng = np.random.default_rng(1)
frames = sol.y.T*(1+0.02*rng.standard_normal((len(frames_t),nx)))

def mlp(nin,w,d,nout):
    L=[]; k=nin
    for _ in range(d): L+=[torch.nn.Linear(k,w),torch.nn.Tanh()]; k=w
    return torch.nn.Sequential(*L,torch.nn.Linear(k,nout)).double()
net_T, net_H = mlp(2,32,3,1), mlp(1,32,3,1)
theta = lambda x,t: net_T(torch.cat([x,t],1))
Hf = lambda x: torch.nn.functional.softplus(net_H(x))
opt = torch.optim.Adam(list(net_T.parameters())+list(net_H.parameters()), lr=2e-3)
x_d = torch.tensor(np.tile(xg,len(frames_t))[:,None]); t_d = torch.tensor(np.repeat(frames_t,nx)[:,None])
y_d = torch.tensor(frames.reshape(-1,1))

t0=time.time()
for it in range(6000):
    opt.zero_grad()
    xc = torch.rand(1500,1,dtype=torch.float64,requires_grad=True)
    tc = torch.rand(1500,1,dtype=torch.float64,requires_grad=True)
    th = theta(xc,tc)
    thx,tht = torch.autograd.grad(th.sum(),(xc,tc),create_graph=True)
    thxx = torch.autograd.grad(thx.sum(),xc,create_graph=True)[0]
    L_pde = ((tht-ALPHA*thxx+Hf(xc)*th)**2).mean()
    xi = torch.rand(300,1,dtype=torch.float64)
    L_ic = ((theta(xi,torch.zeros_like(xi))-1.0)**2).mean()
    xe = torch.tensor([[0.0],[1.0]],dtype=torch.float64,requires_grad=True); te=torch.rand(2,1,dtype=torch.float64)
    the = theta(xe,te); thex = torch.autograd.grad(the.sum(),xe,create_graph=True)[0]
    L_bc = (thex**2).mean()
    L_dat = ((theta(x_d,t_d)-y_d)**2).mean()
    (L_pde+10*L_ic+L_bc+50*L_dat).backward(); opt.step()
H_rec = Hf(torch.tensor(xg[:,None])).detach().numpy().ravel()
l2 = np.linalg.norm(H_rec-H_t)/np.linalg.norm(H_t)
elapsed = time.time()-t0
print(f"PINN field inverse: {elapsed:.0f}s, field rel. L2 error {l2:.3f}")
print(f"delamination at x={xg[np.argmin(H_rec)]:.3f} (true {xg[np.argmin(H_t)]:.3f}); "
      f"min H {H_rec.min():.2f} vs {H_t.min():.2f}; baseline {H_rec.max():.2f} vs {H_t.max():.2f}")

fig, ax = plt.subplots(1,2,figsize=(9,3.4))
for i,tt in enumerate(frames_t): ax[0].plot(xg, frames[i], lw=1, label=f"t={tt}")
ax[0].set_xlabel("x"); ax[0].set_ylabel(r"$\theta$ (surface temp.)"); ax[0].set_title("(a) synthetic IR frames (2% noise)")
ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)
ax[1].plot(xg, H_t, "k-", lw=2, label="true H(x)")
ax[1].plot(xg, H_rec, "C3--", lw=1.8, label="PINN recovered")
ax[1].axvspan(0.52,0.68,color="C3",alpha=0.08); ax[1].set_xlabel("x"); ax[1].set_ylabel("interface conductance H(x)")
ax[1].set_title("(b) recovered delamination field"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
fig.tight_layout(); fig.savefig("figures/electrode_ai/pinn_delamination.pdf"); plt.show()

PINN field inverse: 201s, field rel. L2 error 0.106
delamination at x=0.580 (true 0.600); min H 1.90 vs 0.80; baseline 4.23 vs 4.00


C:\Users\StoerkJulius\AppData\Local\Temp\ipykernel_27560\3633875721.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig("figures/electrode_ai/pinn_delamination.pdf"); plt.show()


**Section 3 result**

The PINN recovered the interface-conductance field in **201 s** with a relative $L_2$ error of **0.106**:

| Quantity | Recovered | True |
|---|---|---|
| delamination location | x = 0.580 | 0.600 |
| baseline conductance | 4.23 | 4.00 |
| dip minimum (defect depth) | 1.90 | 0.80 |

The **localization is excellent** -- the defect centre is found to within one grid spacing, and the healthy-region conductance to ~6%, from four noisy surface-temperature frames that never see $H(x)$ directly. The honest limit is **depth**: the recovered dip (min 1.90) is shallower than the truth (0.80), because a PINN regularizes toward smooth fields and a sharp localized debond is the hardest feature to resolve from diffused surface data. For the manufacturing question this is the right error profile: *where* and *whether* a delamination exists is recovered reliably (the QC-relevant detection), while *how severe* is qualitative without finer frames or a sharper prior. This is a synthetic proof-of-concept, but it demonstrates the one regime the PINN-vs-LSQ benchmark reserved for neural methods -- an unknown **field** with no cheap forward inverse -- and is the concrete entry point for the Part-II flash-thermography QC line on real electrode coatings.

## 4. Summary

The three roadmap items reachable without new measurements all work, and each respects the project's division of labour -- physics closure as backbone, data-driven layer doing only what the closure cannot:

| Method | Role it earns | Result on data already here |
|---|---|---|
| **PySR** | discover the *form*, not fit a number | free search returns a low-order polynomial in $\Pi$ with $\varphi_0\approx0.009$ -- the ansatz class and the VDI-anchored constant, unsupervised |
| **Conformal** | distribution-free intervals that flag transfer limits | 87% mean leave-one-family-out coverage; under-covers exactly the WAIC-flagged NMC811 |
| **PINN** | invert for a *field*, not a scalar | delamination located to one grid point from noisy synthetic IR frames (10.6% field error) |

None of these is "AI for its own sake": each replaces a guess with a discovered form, a point estimate with a calibrated distribution, or a scalar with a field. The remaining roadmap items (recipe$\to$parameter learning, electrode-specific conformal recalibration, the real-electrode flash-thermography inverse) are gated on the partner measurement campaign specified in `publication/measurement_request_partner.md`. The standing recommendation is unchanged: classical methods for the scalar, well-posed problems; these data-driven layers only where the problem structure -- unknown form, needed uncertainty, or unknown field -- actually demands them.